In [15]:

import subprocess
import os
import concurrent.futures

import threading
import sys

def _filter_stderr(pipe):
    noisy_patterns = [
        "MathWorksServiceHost",
        "GLIBCXX_3.4.31",
        "GLIBCXX_3.4.32",
        "CXXABI_1.3.15",
        "libmwflnetwork.so",
        "libmwflcrypto.so",
        "libmwflcryptoutils.so",
        "libmwflcryptoopenssl.so",
        "libmwflurlmgrfactory.so",
    ]

    for line in pipe:
        if not any(p in line for p in noisy_patterns):
            sys.stderr.write(line)
            sys.stderr.flush()

def run(cmd):
    process = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=None,              # leave stdout live
        stderr=subprocess.PIPE    # intercept only stderr
    )

    t = threading.Thread(target=_filter_stderr, args=(process.stderr,))
    t.daemon = True
    t.start()

    returncode = process.wait()
    t.join(timeout=1)

    if returncode != 0:
        print(f"Command exited with code {returncode}", file=sys.stderr)

def lastFrame(directory):
    frames = [int(file.split("geo")[1].split(".")[0]) for file in os.listdir(directory) if file.startswith("geo") and file.endswith(".mat")]
    return max(frames)

here = "/home/nickbroussinos/Code/evolving_stokes_codex/remesh/"

#Inputs
verbose = 'false'
resume = False
supress_outputs = 0
subdivisions = 18
roughness = .0005
remesh_size = 0;
co = 4 
T = 40000;
gamy = -1
Da = 0
Sd = 1
pnd = 0



param = [0]#CHECK RESUME SETTING!!!




commands = []
for Da in param:
        
       #dt = (1.2e-4)*(pp**(-2.0))
        dt = .001
        #print(dt)
        bb = (pnd/.9)*(-4e-8)/(5e-2)
        run_tag = f"gamy_{gamy:+.2f}_Da_{Da:+.2f}_Sd_{Sd:+.2f}".replace('+', 'p').replace('-', 'm')
        dir_path = here + f"data/ms_batch_data/{run_tag}"
        start = (lastFrame(dir_path)) if resume else 0
        if resume:
            print(f"start: {start}")
        cmd = f"""export LD_PRELOAD=/usr/lib/x86_64-linux-gnu/libstdc++.so.6 && \
    matlab -nodisplay -nosplash -nodesktop -r "cd '{here}'; verbose = {verbose};\
    supress_outputs = {supress_outputs}; dir = '{dir_path}/'; start = {start};\
    p = struct(); p.roughness = {roughness}; p.dt={dt};p.T={T}; p.start={start}; p.subdivisions={subdivisions};p.beta={bb};p.co={co};p.pnd = {pnd};p.remesh_size = {remesh_size};p.gamy={gamy};p.Da={Da};p.Sd={Sd};ms_multi; exit" """
        commands.append(cmd)


max_workers = 6
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(run, cmd) for cmd in commands]
    concurrent.futures.wait(futures)

    


                            < M A T L A B (R) >
                  Copyright 1984-2024 The MathWorks, Inc.
             R2024a Update 3 (24.1.0.2603908) 64-bit (glnxa64)
                                May 2, 2024

 
To get started, type doc.
For product information, visit www.mathworks.com.
 
Save geo1.mat at j =1, eps_f = 0.03973, eps_d = 0.0001022, eps_v = 4.181, total time: 2.2313
Save geo2.mat at j =2, eps_f = 0.002428, eps_d = 4.636e-06, eps_v = 4.181, total time: 4.7734
Save geo3.mat at j =1, eps_f = 0.003864, eps_d = 3.791e-06, eps_v = 4.181, total time: 6.5268
Save geo4.mat at j =1, eps_f = 0.002643, eps_d = 3.693e-06, eps_v = 4.181, total time: 8.2508
Save geo5.mat at j =1, eps_f = 0.002475, eps_d = 3.676e-06, eps_v = 4.181, total time: 10.0401
Save geo6.mat at j =1, eps_f = 0.002449, eps_d = 3.675e-06, eps_v = 4.181, total time: 11.8072
Save geo7.mat at j =1, eps_f = 0.002446, eps_d = 3.678e-06, eps_v = 4.181, total time: 13.5204
Save geo8.mat at j =1, eps_f = 0.002448, eps_

Operation terminated by user during ms_multi (line 305)
 
Operation terminated by user during ms_multi (line 305)
 


Save geo3146.mat at j =1, eps_f = 0.01087, eps_d = 2.334e-05, eps_v = 2.663, total time: 6342.6647



KeyboardInterrupt: 

Command exited with code -2
